# ECGtizer Pipeline Vignette

This notebook walks through the **entire ECGtizer digitization pipeline** step by step,
showing the ECG image transformation at every stage.

We use a real 12-lead ECG PDF from `data/data`. Vector PDFs take the lossless path;
scans automatically fall back to the raster tracer shown stage by stage below.

## Pipeline stages

| # | Stage | Function | Output |
|---|-------|----------|--------|
| 0 | Vector recognition | `extract_vector_ecg` | Lossless leads or raster fallback |
| 1 | PDF to Image | `convert_PDF2image` | RGB image |
| 2 | Noise & Format Detection | `check_noise_type` | TYPE, NOISE |
| 3 | Text Masking | `text_extraction` | Cleaned image |
| 4 | Track Segmentation | `tracks_extraction` | Sub-images per track |
| 5 | Rule Removal | scale-relative morphology | Artifact-reduced tracks |
| 6 | Waveform Extraction | `lead_extraction` | Raw digital signals |
| 7 | Extraction Methods Comparison | lazy / full / fragmented | 3 algorithms |
| 8 | Lead Calibration & Naming | `lead_cutting` | Named leads in µV |
| 9 | 12-Lead ECG Plot | `plot_function` | Final result |
| 10 | Overlay Verification | `overlay_coordinates` | Quality check |
| 11 | Export to XML | `write_xml` | HL7 aECG file |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os, sys

# Ensure the project root is on the path
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "ecgtizer")):
    ROOT = os.path.abspath(os.path.join(ROOT, ".."))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)

from ecgtizer.PDF2XML import (
    convert_PDF2image, check_noise_type, text_extraction,
    tracks_extraction, lead_extraction, lead_cutting,
    classic_layout, overlay_coordinates,
)
from ecgtizer.PDF2XML_mod import plot_function, write_xml
from ecgtizer.extraction_functions import (
    lazy_extraction, full_extraction, fragmented_extraction,
)
from ecgtizer.vector_extraction import extract_vector_ecg

plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.facecolor"] = "white"

INPUT_PDF = "data/data/BALOGH_Mihaly_450342_12by1_2024-06-05_2024-06-05.pdf"
DPI = 300

vector_result = extract_vector_ecg(INPUT_PDF)
print(f"Input file: {INPUT_PDF}")
print(f"Resolution: {DPI} DPI")
print("Preferred source:", f"lossless vector ({vector_result.layout})" if vector_result else "raster fallback")

---
## Stage 1 — PDF to Image

`convert_PDF2image` uses **poppler** (via `pdf2image`) to rasterize the PDF at the
requested DPI. The result is a list of NumPy RGB arrays, one per page. Rasterisation
is retained here for diagnostics and for scanned PDFs; validated vector traces are used
for the final signal when `vector_result` is available.

In [ ]:
pages, num_pages, success = convert_PDF2image(INPUT_PDF, DPI=DPI)

image_original = np.array(pages[0])

print(f"Success: {success}")
print(f"Pages:   {num_pages}")
print(f"Shape:   {image_original.shape}  (height x width x channels)")
print(f"Dtype:   {image_original.dtype}")

fig, ax = plt.subplots(figsize=(14, 10))
ax.imshow(image_original)
ax.set_title("Stage 1 — Original ECG Image (from PDF)", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## Stage 2 — Noise & Format Detection

`check_noise_type` analyses the image variance to determine:
- **TYPE**: ECG format (`classic`, `kardia`, `wellue`, `apple`)
- **NOISE**: Whether the background is noisy (`True`, `False`, or `0.5` for partial)

In [ ]:
image = image_original.copy()
TYPE, NOISE = check_noise_type(image, DPI=DPI, DEBUG=False)

variance = np.var(image)

print(f"Detected format : {TYPE}")
print(f"Noise detected  : {NOISE}")
print(f"Image variance  : {variance:.0f}")
print()
print("Variance thresholds:")
print("  < 600          → very clean")
print("  600 – 2000     → clean (NOISE=False)")
print("  2000 – 3000    → partial noise (NOISE=0.5)")
print("  > 3000         → noisy (NOISE=True)")

---
## Stage 3 — Text Masking

For classic ECG pages, `text_extraction` deliberately preserves the source image.
Destructive contour masking can mistake a calibration pulse or QRS tip for text.
Track geometry and continuity handle labels downstream; format-specific masking is
still available for non-classic, low-resolution inputs.

In [ ]:
# Keep a copy before masking
image_before_mask = image.copy()

image_clean = text_extraction(image, page=0, DPI=DPI, NOISE=NOISE, TYPE=TYPE, DEBUG=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].imshow(image_before_mask)
axes[0].set_title("Before text masking", fontsize=13)
axes[0].axis("off")

axes[1].imshow(image_clean)
axes[1].set_title("After text masking", fontsize=13)
axes[1].axis("off")

fig.suptitle("Stage 3 — Text Extraction & Masking", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Show the difference (what was masked)
diff = np.abs(image_before_mask.astype(float) - image_clean.astype(float)).sum(axis=2)
if diff.max() > 0:
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.imshow(diff, cmap="hot")
    ax.set_title("Masked regions (difference map)", fontsize=13)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

---
## Stage 4 — Track Segmentation

`tracks_extraction` finds repeated calibration-pulse edges and fits the standard
4-, 6- or 12-row baseline model. A projection model is the fallback. Each returned
track is an overlapping corridor carrying its exact page origin, target baseline,
neighbour baselines, waveform bounds and calibration geometry.

In [ ]:
dic_tracks, peaksh, peaksv = tracks_extraction(
    image_clean, TYPE, DPI=DPI, FORMAT="", NOISE=NOISE, DEBUG=False
)

print(f"Number of tracks: {len(dic_tracks)}")
print(f"Horizontal peaks (row positions): {list(peaksh)}")
print(f"Vertical signal start (column):   {peaksv}")
print()
for tid, track_img in dic_tracks.items():
    print(f"  Track {tid}: shape {track_img.shape}")

In [ ]:
import cv2

# Recompute row/column ink projections for visualisation
gray = cv2.cvtColor(image_clean, cv2.COLOR_BGR2GRAY)
_, image_bin_viz = cv2.threshold(
    cv2.GaussianBlur(gray, (5, 5), 0), 0, 255,
    cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU,
)
horizontal_variance = (image_bin_viz > 0).sum(axis=1)
vertical_variance = (image_bin_viz > 0).sum(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Horizontal variance
axes[0].plot(horizontal_variance, range(len(horizontal_variance)))
for p in peaksh:
    axes[0].axhline(p, color="red", linewidth=0.8, alpha=0.7)
axes[0].invert_yaxis()
axes[0].set_xlabel("Foreground pixels")
axes[0].set_ylabel("Row (pixels)")
axes[0].set_title("Row projection with fitted baselines")

# Column projection
axes[1].plot(vertical_variance)
axes[1].axvline(peaksv, color="red", linewidth=1, label=f"Signal start: col {peaksv}")
axes[1].set_xlabel("Column (pixels)")
axes[1].set_ylabel("Foreground pixels")
axes[1].set_title("Column projection (active region)")
axes[1].legend()

fig.suptitle("Stage 4 — Geometric Track Detection", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Show each track as a separate subplot
n_tracks = len(dic_tracks)
fig, axes = plt.subplots(n_tracks, 1, figsize=(16, 3 * n_tracks))
if n_tracks == 1:
    axes = [axes]

for idx, (tid, track_img) in enumerate(dic_tracks.items()):
    if len(track_img.shape) == 3:
        axes[idx].imshow(track_img)
    else:
        axes[idx].imshow(track_img, cmap="gray")
    axes[idx].set_title(f"Track {tid}  ({track_img.shape[0]} x {track_img.shape[1]} px)", fontsize=12)
    axes[idx].axis("off")

fig.suptitle("Stage 4 — Segmented Tracks", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Stage 5 — Binarized Tracks

After segmentation, the tracks are already **binarized** (white pixels = signal,
black = background). This is the input to the waveform extraction algorithms.

> **Note**: page-spanning horizontal and vertical print rules are removed with
> scale-relative morphology. Labels remain as competing fragments: deleting them
> by component size also deletes isolated QRS tips. The continuity optimiser resolves
> those fragments inside overlapping, baseline-aware track corridors.


In [ ]:
# Show binarized tracks
# tid = list(dic_tracks.keys())[0]
# track_img = dic_tracks[tid]

# fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# # Original image region (approximate) for comparison
# h = image_original.shape[0]
# track_h = track_img.shape[0]
# # Rough vertical region of this track
# y_start = max(0, peaksh[0] - track_h)
# y_end = min(h, y_start + track_h)
# axes[0].imshow(image_original[y_start:y_end, :], aspect="auto")
# axes[0].set_title(f"Original image (rows {y_start}-{y_end})", fontsize=12)
# axes[0].axis("off")

# # Binarized track
# if len(track_img.shape) == 3:
#     axes[1].imshow(track_img)
# else:
#     axes[1].imshow(track_img, cmap="gray")
# axes[1].set_title(f"Track {tid} — Binarized ({track_img.shape[0]} x {track_img.shape[1]} px)", fontsize=12)
# axes[1].axis("off")

# fig.suptitle("Stage 5 — Binarized Track vs Original", fontsize=14, y=1.02)
# plt.tight_layout()
# plt.show()

# Show all tracks stacked in a single plot
fig, ax = plt.subplots(figsize=(16, 3 * len(dic_tracks)))

y_offset = 0
for tid, track_img in dic_tracks.items():
    h, w = track_img.shape[0], track_img.shape[1]
    extent = [0, w, y_offset + h, y_offset]  # place this track below the previous one

    if len(track_img.shape) == 3:
        ax.imshow(track_img, extent=extent)
    else:
        ax.imshow(track_img, cmap="gray", extent=extent)

    ax.text(-10, y_offset + h / 2, f"Track {tid}", fontsize=12,
             va="center", ha="right")

    y_offset += h

ax.set_xlim(0, max(t.shape[1] for t in dic_tracks.values()))
ax.set_ylim(y_offset, 0)  # invert y-axis so first track is on top
ax.axis("off")
ax.set_title("Stage 5 — Binarized Track Corridors", fontsize=14)

plt.tight_layout()
plt.show()

---
## Stage 6 — Waveform Extraction (Binarization + Pixel-to-Signal)

`lead_extraction` binarizes each track and applies the selected extraction
algorithm to convert pixel positions into a 1-D signal. The signal is then
time-scaled to 5000 waveform samples plus a legacy 140-sample prefix.

Returns:
- `dic_extracted`: scaled signals (standard sample count)
- `dic_image_bin`: binarized track images
- `dic_not_scaled`: raw pixel-level signals before time scaling

In [ ]:
dic_extracted, dic_image_bin, dic_not_scaled = lead_extraction(
    dic_tracks, "trace", TYPE, NOISE=NOISE, DEBUG=False
)

print("Extracted signals:")
for tid in dic_extracted:
    raw_len = len(dic_not_scaled[tid])
    scaled_len = len(dic_extracted[tid])
    print(f"  Track {tid}: {raw_len} px  ->  {scaled_len} samples (time-scaled)")

In [ ]:
# Show binarized track + extracted waveform overlay
n = min(3, len(dic_tracks))
fig, axes = plt.subplots(n, 1, figsize=(16, 3.5 * n))
if n == 1:
    axes = [axes]

for idx, tid in enumerate(list(dic_tracks.keys())[:n]):
    bin_img = dic_image_bin[tid]
    signal_raw = dic_not_scaled[tid]
    local_x = np.arange(len(signal_raw)) + bin_img.waveform_x0

    axes[idx].imshow(bin_img, cmap="gray", aspect="auto")
    axes[idx].plot(local_x, signal_raw, color="red", linewidth=0.8, alpha=0.8)
    axes[idx].set_title(f"Track {tid} — Binarized image + extracted waveform (red)", fontsize=12)
    axes[idx].set_xlim(0, bin_img.shape[1])

fig.suptitle("Stage 6 — Waveform Extraction (trace method)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Stage 7 — Compare Extraction Methods

ECGtizer provides three extraction algorithms with different trade-offs:

| Method | Speed | Accuracy | Description |
|--------|-------|----------|-------------|
| **lazy** | Fast | Moderate | Follows nearest lit pixel from anchor |
| **full** | Fast | Low when lanes overlap | Averages every lit fragment per column |
| **trace** | CPU, global | Highest raster fidelity | Second-order continuity with baseline identity |

In [ ]:
# Compare all three methods on the first track
tid = list(dic_image_bin.keys())[0]
bin_img = dic_image_bin[tid]
waveform_img = bin_img[:, bin_img.waveform_x0:bin_img.waveform_x1]

sig_lazy = lazy_extraction(waveform_img)
sig_full = full_extraction(waveform_img)
sig_frag = fragmented_extraction(waveform_img)

fig, axes = plt.subplots(3, 1, figsize=(16, 9), sharex=True)
methods = [(sig_lazy, "Lazy", "#2196F3"),
           (sig_full, "Full", "#4CAF50"),
           (sig_frag, "Trace", "#F44336")]

for ax, (sig, name, color) in zip(axes, methods):
    ax.imshow(waveform_img, cmap="gray", aspect="auto", alpha=0.4)
    ax.plot(sig, color=color, linewidth=0.9, label=name)
    ax.set_xlim(0, waveform_img.shape[1])
    ax.legend(loc="upper right", fontsize=11)
    ax.set_ylabel("Row (px)")

axes[-1].set_xlabel("Column (px)")
fig.suptitle("Stage 7 — Extraction Methods Comparison (Track 0)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Stage 8 — Lead Calibration & Naming

`lead_cutting` performs two critical operations:

1. **Amplitude calibration**: extracts the reference pulse (1 mV calibration
   square) at the start of each track to determine the pixel-to-microvolt
   conversion factor.

2. **Lead segmentation**: cuts each track into named leads based on the
   detected 3x4, 6x2 or 12x1 layout. When a validated vector result exists,
   its already separate and calibrated source paths are preferred here.

In [ ]:
raster_lead = lead_cutting(
    dic_extracted, DPI, TYPE, FORMAT="", page=0, NOISE=NOISE, DEBUG=False,
    dic_image_bin=dic_image_bin,
)
dic_lead = vector_result.leads if vector_result else raster_lead
source = "lossless vector" if vector_result else "raster trace"

print(f"Detected layout: {classic_layout(len(dic_tracks))}")
print(f"Final source: {source}")
print(f"Number of leads: {len(dic_lead)}")
print()
print(f"{'Lead':<6} {'Samples':>8} {'Min (uV)':>10} {'Max (uV)':>10}")
print("-" * 36)
for name, signal in dic_lead.items():
    print(f"{name:<6} {len(signal):>8} {min(signal):>10.0f} {max(signal):>10.0f}")

In [ ]:
# Show measured calibration geometry for track 0
track0 = dic_image_bin[0]
baseline = track0.baseline_row
gain = track0.calibration_height
if track0.calibration_side == "left":
    pulse_slice = slice(0, max(1, track0.waveform_x0))
else:
    pulse_slice = slice(track0.waveform_x1, track0.shape[1])

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].imshow(track0[:, pulse_slice], cmap="gray", aspect="auto")
axes[0].axhline(baseline, color="blue", linestyle="--", label="0 mV")
axes[0].axhline(baseline - gain, color="red", linestyle="--", label="1 mV")
axes[0].set_title(f"{track0.calibration_side.title()} calibration pulse — {gain:.0f} px/mV")
axes[0].legend(fontsize=10)

axes[1].plot(dic_not_scaled[0], "k-", linewidth=0.5)
axes[1].axhline(baseline, color="blue", linestyle="--", alpha=0.6)
axes[1].set_title("Extracted waveform (calibration excluded)")
axes[1].set_xlabel("Raster column")
axes[1].invert_yaxis()

fig.suptitle("Stage 8 — Amplitude Calibration", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## Stage 9 — Final 12-Lead ECG Plot

The digitized leads are now calibrated in microvolts. We plot them using
`plot_function` which renders a standard 12-lead ECG grid.

In [ ]:
plot_function(dic_lead)

In [ ]:
# Individual lead plots
leads_to_show = ["I", "II", "V1", "V5"]
fig, axes = plt.subplots(len(leads_to_show), 1, figsize=(14, 2.5 * len(leads_to_show)), sharex=True)

for ax, name in zip(axes, leads_to_show):
    if name in dic_lead:
        signal = dic_lead[name]
        t = np.arange(len(signal)) / 500  # time in seconds
        ax.plot(t, signal, "k-", linewidth=0.6)
        ax.set_ylabel(f"{name} (uV)")
        ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Time (s)")
fig.suptitle("Stage 9 — Selected Digitized Leads", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

---
## Stage 10 — Overlay Verification

We overlay the extracted waveforms (red) on the original ECG image to
visually verify extraction quality. The waveforms should follow the
printed traces.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
ax.imshow(image_original, cmap="gray")

# Exact page coordinates come from the geometry carried by each track/signal.
for tid in dic_not_scaled:
    x, y = overlay_coordinates(dic_tracks[tid], dic_not_scaled[tid])
    ax.plot(x, y, color="red", linewidth=0.7, alpha=0.85)

# A vector source is an exact reference overlay because its paths never merged.
if vector_result:
    for source_path in vector_result.paths.values():
        ax.plot(source_path.x * DPI / 72, source_path.y * DPI / 72,
                color="cyan", linewidth=0.45, alpha=0.7)

title = "red: raster trace" + ("; cyan: lossless vector source" if vector_result else "")
ax.set_title(f"Stage 10 — Exact-coordinate overlay ({title})", fontsize=14)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## Stage 11 — Export to HL7 aECG XML

`write_xml` serializes the digitized leads into an HL7 aECG XML file,
the standard format for digital ECG storage.

In [ ]:
os.makedirs("output", exist_ok=True)
xml_path = "output/vignette_00009.xml"

# Metadata table (minimal)
table = {
    "BPM": "75", "low_freq": "0.05", "high_freq": "150",
    "Inter PR (ms)": "160", "Dur.QRS (ms)": "90",
    "QT (ms)": "380", "QTc (ms)": "410",
    "Axe P": "60", "Axe R": "30", "Axe T": "45",
    "Moy RR (ms)": "800", "QTcB (ms)": "405", "QTcF (ms)": "400",
    "Rythme": "Sinus", "ECG": "Normal",
    "Age": "60", "sex": "M", "other_information": "Vignette example",
}

write_xml(dic_lead, xml_path, TYPE=TYPE, table=table)
print(f"XML written to: {xml_path}")
print(f"File size: {os.path.getsize(xml_path):,} bytes")

---
## Stage 12 — Re-render as PDF

As a final step, we can re-render the digitized signals back to a
publication-quality PDF using `xml_to_pdf`.

In [ ]:
from ecgtizer.XML2PDF import xml_to_pdf

pdf_path = "output/vignette_00009.pdf"
xml_to_pdf(xml_path, pdf_path, type_of_pdf="type1")
print(f"PDF written to: {pdf_path}")
print(f"File size: {os.path.getsize(pdf_path):,} bytes")
print("\nOpen the PDF to see the re-rendered 12-lead ECG.")

---
## Summary

| Stage | Function | Input | Output |
|-------|----------|-------|--------|
| 0 | `extract_vector_ecg` | PDF file | Lossless leads or `None` fallback |
| 1 | `convert_PDF2image` | PDF file | RGB image (NumPy array) |
| 2 | `check_noise_type` | Image | TYPE (str), NOISE (bool/float) |
| 3 | `text_extraction` | Image | Masked image (text regions whitened) |
| 4 | `tracks_extraction` | Masked image | Dict of track sub-images |
| 5 | scale-relative rule removal | Binary page | Artifact-reduced track corridors |
| 6 | `lead_extraction` | Track images | 1-D signals (raw + time-scaled) |
| 7 | lazy/full/trace | Binary track | Comparison of 3 algorithms |
| 8 | `lead_cutting` | Scaled signals | Named leads calibrated in uV |
| 9 | `plot_function` | Lead dict | 12-lead ECG grid plot |
| 10 | overlay | Original image + signals | Visual verification |
| 11 | `write_xml` | Lead dict | HL7 aECG XML file |
| 12 | `xml_to_pdf` | XML file | Publication-quality PDF |

For more details, see the [API Reference](api/index.rst) documentation.